In [188]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [189]:
!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

  Using cached scikit-learn-1.2.2.tar.gz (7.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [595 lines of output]
      Partial import of sklearn during the build process.
      test_program.c
      Generating code
      Finished generating code
      test_program.c
      LINK : warning LNK4044: unrecognized option '/openmp'; ignored
      Generating code
      Finished generating code
      Compiling sklearn\__check_build\_check_build.pyx because it changed.
      Compiling sklearn\_isotonic.pyx because it changed.
      Compiling sklearn\_loss\_loss.pyx because it changed.
      Compiling sklearn\cluster\_dbscan_inner.pyx because it changed.
      Compiling sklearn\cluster\_hierarchical_fast.pyx because it changed.
      Compiling sklearn\cluster\_k_means_common.pyx because it changed.
      Compiling sklearn\cluster\_k_means_lloyd.pyx because it changed.
      Compiling sklearn\cluster\_k_means_elkan.pyx because it changed.
      Compiling sklea

'1.6.1'

In [190]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from sklearn.metrics import davies_bouldin_score
from collections import Counter


In [191]:
pypfopt.__version__

'1.5.5'

In [192]:
!pip install --upgrade yfinance
import yfinance as yf
import pandas as pd


In [193]:
from pandas_datareader import data as pdr

In [194]:
def modify_date(date):
    return date[:10]

In [195]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [196]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [197]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [198]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [199]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [200]:
def output(weight):
    if stock_type == "S&P100":
      OEX=yf.download("^OEX", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]  # YFINANCE_FIX: added auto_adjust=False

    elif stock_type == "S&P500":
      OEX=yf.download("SPY", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]  # YFINANCE_FIX: added auto_adjust=False

    stock_price=yf.download(list(weight.index), start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]  # YFINANCE_FIX: added auto_adjust=False
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX.iloc[0,0])
    injected_result["oex"]=S2*OEX
    return injected_result

## 1. Preselection

In [201]:
# Preselection parameters set up
#method = "K_means_multiple"# or avg or anyother method
method = "K_means_multiple"
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2020,2,18)
end_swan_date = dt(2021,3,23) 

w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means
Min_appear = 3
Seeds = 5

In [202]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100_MVO2.xlsx",index_col=0)
    #portfolio_beta = pd.read_excel("BetaMatrix S&P100 Cathy(Yijue) 2022.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("SP500_BETALouise 8_30_24 (2).xlsx",index_col=0)

    optdata = pd.read_excel("optdata.xlsx",index_col=0,sheet_name = 1)
    portfolio_beta = portfolio_beta[portfolio_beta.index.isin(optdata.index)]

In [203]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [204]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,-0.004268,0.111982,0.050073,0.001444,0.101963,0.079685,0.076826
Swan2,NaN,1.000000,0.549511,0.335485,0.410861,0.392490,0.033937,0.307858
Swan3,NaN,NaN,1.000000,0.151778,0.258652,0.435457,0.243705,0.224528
Swan4,NaN,NaN,NaN,1.000000,-0.032138,0.143125,0.147027,0.225411
Swan5,NaN,NaN,NaN,NaN,1.000000,0.444474,0.221512,0.208068
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.301317,0.482711
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.185835
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [205]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
## Uncomented out usually, but changes results
#portfolio_beta["corr_beta"]=selected_beata@weight_swan 
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [206]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,Seeds):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    stock_frequency_name = f"Astock_frequency_w1_{w1}_Seeds_{Seeds}.csv"
    df.to_csv(stock_frequency_name)

    for p in range(0,min(selected_stock_number,len(sorted_count))):
      if sorted_count[p][1]>=Min_appear:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

elif method == "Custom":
    # Step 1: Calculate Davies-Bouldin index for random seeds
    selected_beta = portfolio_beta.loc[:, start_swan:end_swan].dropna()
    dbi_scores = []

    # Loop through random seeds from 1 to 1000 to find the best seeds with lowest DBI
    for R in range(1, 1001):
        kmeans = KMeans(n_clusters=N, random_state=R)
        labels = kmeans.fit_predict(selected_beta)
        dbi = davies_bouldin_score(selected_beta, labels)
        dbi_scores.append((R, dbi))

    # Sort by Davies-Bouldin index and select the top 5 seeds with lowest DBI
    top_seeds = sorted(dbi_scores, key=lambda x: x[1])[:Seeds]


    stock_counter = Counter()

    for R, _ in top_seeds:

        selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
        Y_predic = selected_beta[end_swan].copy()
        Kmean = KMeans(n_clusters=N,random_state=R)#set the random_state to keep modeling the same every run
        Kmean.fit(selected_beta)
        ct = Kmean.cluster_centers_
        Kmean.labels_
        selected_beta["Group"]=Kmean.labels_
        res = ct.mean(axis=1)
        res [res<0]=0

        min_group = np.where(res==min(res))
        for i in range(N):
            Y_predic[selected_beta["Group"] == i] = res [i]
        selected_beta ["avg"]=Y_predic.values
        selected_beta=selected_beta.sort_values(by=["avg"])
        first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
        if first_group.shape[0] <10:
            second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
            preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=True)
        else:
            preselected_stock = first_group

        stock_counter.update(preselected_stock.index)

    top_stocks = [stock for stock, count in stock_counter.most_common(22)]

c:\Users\ganpo\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ganpo\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ganpo\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ganpo\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window

['PFE', 'AMGN', 'KO', 'CL', 'VZ', 'COST', 'MO', 'MRK', 'JNJ', 'SBUX', 'NEE', 'PEP', 'SO', 'LLY', 'ABT', 'AMZN', 'EXC', 'MDLZ', 'WMT', 'PG', 'PFE', 'AMGN', 'KO', 'CL', 'VZ', 'COST', 'MO', 'MRK', 'JNJ', 'SBUX', 'NEE', 'PEP', 'SO', 'LLY', 'ABT', 'AMZN', 'EXC', 'MDLZ', 'WMT', 'PG', 'PFE', 'AMGN', 'KO', 'CL', 'VZ', 'COST', 'MO', 'MRK', 'JNJ', 'SBUX', 'NEE', 'PEP', 'SO', 'LLY', 'ABT', 'AMZN', 'EXC', 'MDLZ', 'WMT', 'PG', 'WMT', 'CL', 'ABT', 'MDLZ', 'VZ', 'PG', 'EXC', 'SO', 'JNJ', 'KO', 'LLY', 'SBUX', 'PEP', 'NEE', 'MRK', 'VZ', 'NEE', 'PEP', 'SO', 'PG', 'MRK', 'ABT', 'SBUX', 'JNJ', 'KO', 'MDLZ', 'LLY', 'WMT', 'CL', 'EXC']
[('ABT', 5), ('CL', 5), ('EXC', 5), ('JNJ', 5), ('KO', 5), ('LLY', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 5), ('WMT', 5), ('AMGN', 3), ('AMZN', 3), ('COST', 3), ('MO', 3), ('PFE', 3)]
           Swan4     Swan5     Swan6     Swan7     Swan8  Group       avg
Ticker                                                          

C:\Users\ganpo\AppData\Local\Temp\ipykernel_25308\3798134386.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]


In [207]:
preselected_stock


,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg
Ticker,,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,0,0.598607
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,1,0.582198
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,4,0.380388
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,0,0.405164
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,1,0.665586
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,0,0.411908
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,0,0.545491
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,0,0.251446
LLY,0.746268,0.920577,0.280671,0.780342,0.061466,0,0.557865


In [208]:
# #holt step
# portfolio_beta = pd.read_csv("HoltestimateS9.csv",index_col=0)
# if 'Ticker' in preselected_stock.columns:
#     preselected_stock.set_index('Ticker', inplace=True)
# preselected_stock['avg'] = preselected_stock.index.map(portfolio_beta['Forecasted Swan9'])


In [ ]:
#using average step

preselected_stock

# preselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5
preselected_stock['avg'] = portfolio_beta.loc[preselected_stock.index, 'Swan9']


In [210]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR
Ticker,,,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,0,0.598607,0.401393
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,1,0.582198,0.417802
AMZN,-0.445606,0.328167,0.123350,0.334033,1.561997,4,0.380388,0.619612
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,0,0.405164,0.594836
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,1,0.665586,0.334414
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,0,0.411908,0.588092
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,0,0.545491,0.454509
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,0,0.251446,0.748554
LLY,0.746268,0.920577,0.280671,0.780342,0.061466,0,0.557865,0.442135


## Step 2 - Download data and adjusted price based on Swan Beta


In [211]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['ABT' 'AMGN' 'AMZN' 'CL' 'COST' 'EXC' 'JNJ' 'KO' 'LLY' 'MDLZ' 'MO' 'MRK'
 'NEE' 'PEP' 'PFE' 'PG' 'SBUX' 'SO' 'VZ' 'WMT']


In [212]:
yahooDataFrame= yf.download(list(selected_stock_list), start=start, end=end, auto_adjust=False)["Close"] 
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

[*********************100%***********************]  20 of 20 completed


Ticker,ABT,AMGN,AMZN,CL,COST,EXC,JNJ,KO,LLY,MDLZ,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,
2019-01-02,69.500000,191.899994,76.956497,59.250000,204.759995,31.433666,127.750000,46.930000,114.879997,39.889999,49.310001,72.127861,42.457500,109.279999,41.034157,91.279999,64.320000,43.720001,56.020000,31.113333
2019-01-03,66.220001,188.979996,75.014000,58.970001,200.419998,31.469330,125.720001,46.639999,111.309998,39.740002,49.040001,70.648857,42.352501,108.260002,39.886147,90.639999,61.529999,44.340000,56.220001,30.953333
2019-01-04,68.110001,195.440002,78.769501,59.520000,206.240005,31.825962,127.830002,47.570000,114.660004,40.840000,50.299999,72.776718,43.132500,110.480003,40.796963,92.489998,63.570000,44.709999,56.360001,31.146667
2019-01-07,69.129997,198.070007,81.475502,59.880001,207.000000,31.883024,127.010002,46.950001,115.279999,41.119999,48.779999,71.975189,43.070000,109.529999,41.015179,92.120003,63.570000,44.610001,56.720001,31.513332
2019-01-08,68.300003,200.610001,82.829002,60.549999,208.550003,32.375179,129.960007,47.480000,116.339996,42.080002,49.060001,72.509544,43.384998,110.580002,41.204933,92.459999,63.619999,45.880001,58.380001,31.733334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,88.300003,227.419998,106.695503,77.349998,313.790009,34.457916,151.860001,59.750000,145.509995,58.970001,46.200001,81.736641,67.934998,145.660004,35.872864,126.169998,87.529999,68.440002,60.240002,38.416668
2020-02-11,89.169998,228.960007,107.540001,76.110001,310.679993,35.192581,151.990005,59.590000,144.250000,58.869999,45.570000,81.345421,68.037498,146.080002,36.138519,124.160004,87.510002,68.589996,58.689999,38.466667
2020-02-12,88.790001,227.000000,108.000000,75.949997,315.119995,35.135521,151.089996,59.410000,143.509995,58.950001,45.400002,79.437019,67.565002,146.080002,35.806454,123.449997,88.570000,68.620003,58.230000,38.616669


In [213]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [214]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


In [215]:
market_drop.to_csv("market_drop.csv")

In [216]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMGN', 'AMZN', 'CL', 'COST', 'EXC', 'JNJ', 'KO', 'LLY',
       'MDLZ', 'MO', 'MRK', 'NEE', 'PEP', 'PFE', 'PG', 'SBUX', 'SO', 'VZ',
       'WMT'], dtype=object)

In [217]:
yahooDataFrame

Ticker,ABT,AMGN,AMZN,CL,COST,EXC,JNJ,KO,LLY,MDLZ,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,,,
2019-01-02,69.500000,191.899994,76.956497,59.250000,204.759995,31.433666,127.750000,46.930000,114.879997,39.889999,49.310001,72.127861,42.457500,109.279999,41.034157,91.279999,64.320000,43.720001,56.020000,31.113333
2019-01-03,66.073334,188.572908,74.908423,58.881599,199.926430,31.421369,125.466258,46.596608,111.080243,39.690526,48.952030,70.490708,42.305515,108.065924,39.784562,90.513879,61.433032,44.332966,56.133476,30.891961
2019-01-04,67.808294,194.597995,78.547775,59.341547,205.224204,31.728953,127.313999,47.481486,114.186664,40.738309,50.119537,72.450893,43.036797,110.083888,40.589155,92.232609,63.369637,44.695814,56.186520,31.023157
2019-01-07,68.670661,196.790000,81.131488,59.610702,205.470683,31.737249,126.240962,46.818961,114.566151,40.966416,48.517485,71.491834,42.926654,108.940935,40.701799,91.735465,63.269456,44.588771,56.458118,31.325886
2019-01-08,67.694907,198.881438,82.362697,60.186916,206.495645,32.177812,128.910804,47.303307,115.379447,41.870444,48.707974,71.860286,43.192473,109.787054,40.785161,91.945389,63.218958,45.850889,58.020604,31.481660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,88.300003,227.419998,106.695503,77.349998,313.790009,34.457916,151.860001,59.750000,145.509995,58.970001,46.200001,81.736641,67.934998,145.660004,35.872864,126.169998,87.529999,68.440002,60.240002,38.416668
2020-02-11,89.169998,228.960007,107.540001,76.110001,310.679993,35.192581,151.990005,59.590000,144.250000,58.869999,45.570000,81.345421,68.037498,146.080002,36.138519,124.160004,87.510002,68.589996,58.689999,38.466667
2020-02-12,88.790001,227.000000,108.000000,75.949997,315.119995,35.135521,151.089996,59.410000,143.509995,58.950001,45.400002,79.437019,67.565002,146.080002,35.806454,123.449997,88.570000,68.620003,58.230000,38.616669


# Step 3: MVO

In [218]:
Portfolio_Date=end
data_cleaning(yahooDataFrame)


In [219]:
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]

stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]

In [220]:
daily_return = yahooDataFrame.pct_change().dropna()
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [221]:
opt = pd.read_excel('optdata_ganesh_4.xlsx', index_col=0).loc[preselected_stock.index]
R_fwd = opt['R_fwd_30_MA']
fall = pd.Series(max_DD, index = preselected_stock.index)

In [222]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = R_fwd/fall


# Step 4: Optimization

In [223]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.00)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)

OrderedDict({'ABT': 0.0, 'AMGN': 0.0, 'AMZN': 0.17, 'CL': 0.0, 'COST': 0.17, 'EXC': 0.04266, 'JNJ': 0.0, 'KO': 0.0, 'LLY': 0.07963, 'MDLZ': 0.17, 'MO': 0.02771, 'MRK': 0.17, 'NEE': 0.0, 'PEP': 0.0, 'PFE': 0.0, 'PG': 0.0, 'SBUX': 0.0, 'SO': 0.0, 'VZ': 0.0, 'WMT': 0.17})
Expected annual return: 1.0%
Annual volatility: 9.9%
Sharpe Ratio: 0.10


c:\Users\ganpo\anaconda3\Lib\site-packages\pypfopt\efficient_frontier\efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.009711915491158163, 0.09915141682155704, 0.09795034506301319)

In [224]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")
weight

,weight
AMZN,0.174845
COST,0.174845
EXC,0.043876
LLY,0.081899
MDLZ,0.174845
MRK,0.174845
WMT,0.174845


In [225]:
if stock_type == "S&P100":
    OEX=yf.download("^OEX", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"]  

elif stock_type == "S&P500":
    OEX=yf.download("SPY", start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"] 

stock_price=yf.download(list(weight.index), start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"] 

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  7 of 7 completed


In [226]:
stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,AMZN,COST,EXC,LLY,MDLZ,MRK,WMT
Date,,,,,,,
2020-02-18,107.783501,322.049988,35.934380,141.009995,59.660000,78.683205,39.876667
2020-02-19,108.511002,323.019989,35.378033,140.830002,59.299999,78.244278,39.226665
2020-02-20,107.654999,324.079987,35.556347,142.000000,59.040001,78.711830,39.230000
2020-02-21,104.798500,321.950012,35.463623,142.039993,58.750000,78.568703,39.526669
2020-02-24,100.464500,313.619995,34.821682,138.149994,57.740002,77.604965,38.773335
...,...,...,...,...,...,...,...
2021-03-16,154.593002,327.250000,30.927246,190.899994,57.779999,73.291985,44.463333
2021-03-17,156.786499,329.190002,30.649073,186.729996,57.919998,73.759544,44.093334
2021-03-18,151.399506,322.980011,30.456491,183.929993,57.910000,73.730919,43.336666


In [227]:
weight

,weight
AMZN,0.174845
COST,0.174845
EXC,0.043876
LLY,0.081899
MDLZ,0.174845
MRK,0.174845
WMT,0.174845


In [228]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)[0]
beta = stock_drop/(market_drop)
1-beta.T@weight

C:\Users\ganpo\AppData\Local\Temp\ipykernel_25308\1787084993.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)[0]


weight    0.488168
dtype: float64

In [229]:
beta

Ticker
AMZN    0.390752
COST    0.368083
EXC     1.254518
LLY     0.490481
MDLZ    0.928976
MRK     0.603351
WMT     0.091629
dtype: float64

In [230]:
stock_drop

Ticker
AMZN    0.123209
COST    0.116061
EXC     0.395565
LLY     0.154655
MDLZ    0.292917
MRK     0.190244
WMT     0.028892
dtype: float64

In [231]:
injected_result = pd.DataFrame(index = stock_price.index)
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX.iloc[0])
injected_result["oex"]=S2*OEX
OEX

C:\Users\ganpo\AppData\Local\Temp\ipykernel_25308\2967989542.py:5: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S2 = 10000/float(OEX.iloc[0])


Ticker,^OEX
Date,
2020-02-18,1511.119995
2020-02-19,1518.939941
2020-02-20,1511.270020
2020-02-21,1493.890015
2020-02-24,1440.319946
...,...
2021-03-16,1797.390015
2021-03-17,1802.180054
2021-03-18,1772.750000


In [232]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.48822066443083645


C:\Users\ganpo\AppData\Local\Temp\ipykernel_25308\3168416286.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
C:\Users\ganpo\AppData\Local\Temp\ipykernel_25308\3168416286.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2020-02-19,9960.424983,10051.749340
2020-03-23,8353.109717,6882.312479


In [233]:
import yfinance as yf
from datetime import datetime as dt

start = dt(2020, 2, 18)
end   = dt(2020, 3, 24)  # +1 day to include Mar 23

oex = yf.download("^OEX", start=start, end=end, 
                   auto_adjust=False)["Close"]

P_start = oex.iloc[0]   # Feb 18, 2020
P_end   = oex.iloc[-1]  # Mar 23, 2020

investment = 10_000
R = P_end / P_start - 1
final_value = investment * (1 + R)
final_value

[*********************100%***********************]  1 of 1 completed


Ticker
^OEX    6882.312479
dtype: float64

In [234]:
# From the injected_result DataFrame (portfolio_value column)
portfolio_returns = injected_result["portfolio_value"].pct_change().dropna()

# Annualized Sharpe (assuming ~252 trading days)
sharpe = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(252)
print(f"Realized Sharpe Ratio: {sharpe:.4f}")


Realized Sharpe Ratio: 0.4877


In [235]:
Portfolio_DD_L = abs(injected_result['portfolio_value'].iloc[24] / injected_result['portfolio_value'].iloc[1] - 1)
DD_L = abs(injected_result['oex'].iloc[24] / injected_result['oex'].iloc[1] - 1)
PR_L = 1 - (Portfolio_DD_L / DD_L)

print(f"Portfolio Drawdown: {Portfolio_DD_L}")
print(f"DD: {DD_L}")
print(f"Performance Ratio (PR): {PR_L}")

Portfolio Drawdown: 0.16137014921307424
DD: 0.3153119674783472
Performance Ratio (PR): 0.48822066443083645


In [236]:
injected_result_name = f"AInjected_result_w1_{w1}_Seeds_{Seeds}.csv"
injected_result.to_csv(injected_result_name)

In [237]:
gradient_matrix = weight.join(portfolio_beta, how='outer')
gradient_matrix_name = f"AAAgradient_matrix_Weight:_{w1}_Seeds:_{Seeds}.csv"

In [238]:
file_name = f"AAA_gradient_matrix_Weight_{w1}_Seeds:_{Seeds}.xlsx"

with pd.ExcelWriter(file_name) as writer:
    gradient_matrix.to_excel(writer, sheet_name='Sheet1 - Weights and Beta')
    injected_result.to_excel(writer, sheet_name='Sheet2 - injected_result')
    df.to_excel(writer, sheet_name='Sheet3 - frequency')

In [239]:
E_beta = preselected_stock['avg']
swan_beta_matrix = preselected_stock.iloc[:,0:4]
beta_cov_raw = np.cov(swan_beta_matrix.values, rowvar=True, bias=False)
beta_cov_matrix = pd.DataFrame(
    beta_cov_raw + np.eye(len(selected_stock_list)) * 1e-6,
    index=selected_stock_list, columns=selected_stock_list
)
swan_beta_matrix

,Swan4,Swan5,Swan6,Swan7
Ticker,,,,
ABT,0.702188,0.474696,0.311079,1.073150
AMGN,0.795355,0.502892,0.109292,0.843475
AMZN,-0.445606,0.328167,0.123350,0.334033
CL,0.099788,0.617542,-0.045657,0.784583
COST,0.829818,0.976294,-0.015790,0.618966
EXC,0.519667,0.794675,0.263389,0.605246
JNJ,0.403161,0.739963,0.360346,0.668742
KO,-0.170562,0.536675,0.325968,0.614677
LLY,0.746268,0.920577,0.280671,0.780342


In [240]:
# # ============================================================
# # OPTIMIZATION 1: Maximize (1-Beta) / SD(Beta)
# # ============================================================
# from scipy.optimize import minimize
# print("="*60)
# print("OPTIMIZATION 1: Maximize (1-Beta) / SD(Beta)")
# print("="*60)

# n = selected_stock_number
# bounds_opt = [(0, w1)] * n
# constraints_opt = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
# w0_opt = np.ones(n) / n

# E_beta_vals = E_beta.values
# beta_cov_vals = beta_cov_matrix.values
# cov_vals = cov.values
# E_2_vals = (E_2 / 100).values

# def neg_pr_over_sd_beta(w):
#     """Maximize (1-Beta) / SD(Beta)"""
#     port_beta = w @ E_beta_vals
#     port_pr = 1 - port_beta
#     beta_var = w @ beta_cov_vals @ w
#     beta_sd = np.sqrt(max(beta_var, 1e-12))
#     return -port_pr / beta_sd

# res_m1 = minimize(neg_pr_over_sd_beta, w0_opt, method='SLSQP',
#                   bounds=bounds_opt, constraints=constraints_opt)
# weight_m1 = res_m1.x.copy()
# weight_m1[weight_m1 < 1e-4] = 0
# weight_m1 = weight_m1 / weight_m1.sum()

# beta_m1 = weight_m1 @ E_beta_vals
# pr_m1 = 1 - beta_m1
# sd_beta_m1 = np.sqrt(weight_m1 @ beta_cov_vals @ weight_m1)
# sd_ret_m1 = np.sqrt(weight_m1 @ cov_vals @ weight_m1)
# ret_m1 = weight_m1 @ E_2_vals
# sr_m1 = (ret_m1 - 0.01) / sd_ret_m1 if sd_ret_m1 > 0 else 0
# obj1_val = pr_m1 / sd_beta_m1

# wdf_m1 = pd.DataFrame(weight_m1, index=selected_stock_list, columns=['weight'])
# wdf_m1_clean = wdf_m1[wdf_m1.weight > 0.001]
# print(wdf_m1_clean.sort_values('weight', ascending=False))
# print(f"\nPortfolio Beta: {beta_m1:.4f}")
# print(f"PR (1-Beta):    {pr_m1:.4f}")
# print(f"SD(Beta):       {sd_beta_m1:.4f}")
# print(f"Objective:      {obj1_val:.4f}")
# print(f"Sharpe Ratio:   {sr_m1:.4f}")
# print(f"Exp Return:     {ret_m1:.4f}")
# print(f"Volatility:     {sd_ret_m1:.4f}")
# print(f"Optimizer success: {res_m1.success}")

In [241]:
# weight_m1

In [242]:
# weights = pd.DataFrame(weight_m1, index = list(preselected_stock.index),columns = ["weight"])
# weights = weights[weights['weight'] != 0]
# weights.to_csv('weights.csv')

In [243]:
# stock_price=yf.download(list(weights.index), start=start_swan_date, end=end_swan_date, auto_adjust=False)["Close"] 
# stock_price.to_csv("tradeinfo.csv")


In [244]:
# injected_result = pd.DataFrame(index = stock_price.index)
# portfolio_value = np.sum(np.array(weights.T) * stock_price,axis=1)
# S1 = np.array(10000*weights.T)/np.array(stock_price.iloc[0,:]).T
# injected_result["portfolio_value"]=stock_price@S1.T
# S2 = 10000/float(OEX.iloc[0])
# injected_result["oex"]=S2*OEX


In [245]:
# injected_result_name = f"AInjected_result_w1_{w1}_Seeds_{Seeds}.csv"
# injected_result.to_csv(injected_result_name)

In [246]:
# injected_result.head(30)